# Code for generation of synthetic EEG data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_probability as tfp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from google.colab import drive
from scipy.io import savemat
import os

# Mount drive
drive.mount('/content/drive')

# folder in Google Drive
drive_folder = "/content/drive/My Drive/EEG_Simulation/"
control_folder = os.path.join(drive_folder, "Control")
adhd_folder = os.path.join(drive_folder, "ADHD")

# Create folders
os.makedirs(control_folder, exist_ok=True)
os.makedirs(adhd_folder, exist_ok=True)

# Parameters
fs = 128
duration = 30
n_samples = fs * duration
n_signals = 50

# Frequency bands
delta_range = (0.5, 4)
theta_range = (4, 7)
alpha_range = (8, 13)
beta_range = (14, 30)

# Generate EEG-like signal
def generate_eeg_signal(condition="healthy"):
    time = np.linspace(0, duration, n_samples)

    delta_wave = np.sin(2 * np.pi * (np.random.uniform(*delta_range) + np.random.uniform(-0.2, 0.2)) * time)
    theta_wave = np.sin(2 * np.pi * (np.random.uniform(*theta_range) + np.random.uniform(-0.3, 0.3)) * time)
    alpha_wave = np.sin(2 * np.pi * (np.random.uniform(*alpha_range) + np.random.uniform(-0.5, 0.5)) * time)
    beta_wave = np.sin(2 * np.pi * (np.random.uniform(*beta_range) + np.random.uniform(-1, 1)) * time)

    noise_level = np.random.uniform(0.3, 1.0) if condition == "ADHD" else 0.5
    noise = np.random.normal(0, noise_level, n_samples)

    if condition == "ADHD":
        delta_weight = np.random.uniform(1.5, 2.5)
        theta_weight = np.random.uniform(1.8, 2.5)
        alpha_weight = np.random.uniform(0.7, 1.0)
        beta_weight = np.random.uniform(0.3, 0.7)
    else:
        delta_weight = np.random.uniform(0.8, 1.2)
        theta_weight = np.random.uniform(0.8, 1.2)
        alpha_weight = np.random.uniform(0.9, 1.2)
        beta_weight = np.random.uniform(0.9, 1.2)

    eeg_signal = (
        delta_weight * delta_wave +
        theta_weight * theta_wave +
        alpha_weight * alpha_wave +
        beta_weight * beta_wave +
        noise
    )

    if np.random.rand() < 0.3:
        eeg_signal[np.random.randint(0, n_samples, size=10)] += np.random.uniform(2, 4)

    return eeg_signal


def normalize_zscore(signal):
    return (signal - np.mean(signal)) / np.std(signal)

# Generate and save Control subjects
print("Generating Control subjects...")
for i in range(n_signals):
    # Generate signal
    signal = generate_eeg_signal("healthy")

    # Normalize
    signal_normalized = normalize_zscore(signal)

    # Reshape to (n_samples, 1) - rows are time samples, column is the channel
    signal_reshaped = signal_normalized.reshape(-1, 1)

    # Save as .mat file
    filename = f"control_subject_{i+1:03d}.mat"
    filepath = os.path.join(control_folder, filename)
    savemat(filepath, {'data': signal_reshaped})

    if (i + 1) % 10 == 0:
        print(f"  Saved {i + 1}/{n_signals} Control subjects")

print(f"All Control subjects saved to: {control_folder}\n")

# Generate and save ADHD subjects
print("Generating ADHD subjects...")
for i in range(n_signals):
    # Generate signal
    signal = generate_eeg_signal("ADHD")

    # Normalize
    signal_normalized = normalize_zscore(signal)

    # Reshape to (n_samples, 1) - rows are time samples, column is the channel
    signal_reshaped = signal_normalized.reshape(-1, 1)

    # Save as .mat file
    filename = f"adhd_subject_{i+1:03d}.mat"
    filepath = os.path.join(adhd_folder, filename)
    savemat(filepath, {'data': signal_reshaped})

    if (i + 1) % 10 == 0:
        print(f"  Saved {i + 1}/{n_signals} ADHD subjects")

print(f"All ADHD subjects saved to: {adhd_folder}\n")

In [ ]:
from scipy.io import loadmat

# Load a single subject
data = loadmat('/content/drive/My Drive/EEG_Simulation/Control/control_subject_001.mat')
eeg_signal = data['data']  # Shape: (3840, 1)
eeg_signal.shape